# Week 2: Pandas 数据分析的核心 (SQL 杀手)

> **💡 核心目标**：
> 忘掉 `for` 循环！从今天开始，我们要用向量化思维（Vectorization）处理数据。
> 本代码本包含了 Week 2 前 3 天的所有核心内容：**查 (Select)**、**滤 (Where)**、**聚 (Group By)**。

In [2]:
# 🛠️ 准备工作：导入 Pandas
# 习惯上我们将 pandas 简写为 pd
import pandas as pd
import numpy as np
import seaborn as sns  # 借用它来下载真实数据集

print("Pandas environment is ready!")

Pandas environment is ready!


## 🟢 Part 1: DataFrame 的创建与透视 (Pandas 101)

**对应 SQL**：`CREATE TABLE` + `INSERT INTO`

### 练习 1.1：手搓一个 DataFrame
我们先手动创建一个包含 `Product`, `Price`, `Sales` 的简单表格。

In [3]:
# 字典转 DataFrame (最常用)
data = {
    'Product': ['Apple', 'Banana', 'Cherry', 'Dates'],
    'Price': [5.0, 3.0, 10.0, 15.0],
    'Sales': [100, 200, 50, 20]
}

df = pd.DataFrame(data)
df

,Product,Price,Sales
0,Apple,5.0,100
1,Banana,3.0,200
2,Cherry,10.0,50
3,Dates,15.0,20


### 练习 1.2：加载真实数据
我们将使用经典数据集 `tips` (餐厅小费数据)。
*   `total_bill`: 账单总额
*   `tip`: 小费
*   `sex/smoker/day/time`: 各种分类标签
*   `size`: 用餐人数

In [4]:
# 直接从云端加载数据
tips = sns.load_dataset('tips')

# 👀 看看前 5 行 (SQL: LIMIT 5)
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


### 练习 1.3：数据“体检”
拿到数据第一件事：看有多少行？有没有空值？统计分布是啥？

In [12]:
# ✏️ 请在下面尝试运行：
# 1. tips.info()      <-- 看字段类型
# 2. tips.describe()  <-- 看平均值、最大最小值 (神器!)
# 3. tips.shape       <-- 看(行数, 列数)

# tips.info()
# tips.describe()
# tips.shape
print("=== INFO ===")
tips.info()  # 让它自己打印

print("\n=== SHAPE ===")
print(tips.shape) # shape 是个属性，不是函数，得 print 出来

print("\n=== DESCRIBE ===")
print(tips.describe()) # describe 返回的是表格，也可以 print

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB

=== SHAPE ===
(244, 7)

=== DESCRIBE ===
       total_bill         tip        size
count  244.000000  244.000000  244.000000
mean    19.785943    2.998279    2.569672
std      8.902412    1.383638    0.951100
min      3.070000    1.000000    1.000000
25%     13.347500    2.000000    2.000000
50%     17.795000    2.900000    2.000000
75%     24.127500    3.562500    3.000000
max     50.810000   10.000000    6.000000


## 🟡 Part 2: 查询与过滤 (The WHERE Clause)

**对应 SQL**：`SELECT ... WHERE ...`

### 练习 2.1：列选择
只要 `total_bill` 和 `tip` 这两列。

In [15]:
# 单列 (返回 Series)
# tips['total_bill']

# 多列 (返回 DataFrame，注意是双层中括号 [[...]])
# tips[['total_bill', 'tip']]

# ✏️ 动手试试：只选出 'day' 和 'size' 两列
tips[['day','size']]


,day,size
0,Sun,2
1,Sun,3
2,Sun,3
3,Sun,2
4,Sun,4
...,...,...
239,Sat,3
240,Sat,2
241,Sat,2
242,Sat,2


### 练习 2.2：条件查询 (最重要！)
找出 **晚餐 (Dinner)** 且 **小费 > 5美元** 的慷慨食客。

*   SQL: `WHERE time = 'Dinner' AND tip > 5`
*   Pandas: `df[(条件1) & (条件2)]` (注意 AND 是 `&`，OR 是 `|`)

In [23]:
# ✏️ 请补全代码：
rich_dinners = tips[ (tips['time'] == 'Dinner') & (tips['tip'] > 5) ]
rich_dinners

,total_bill,tip,sex,smoker,day,time,size
23,39.42,7.58,Male,No,Sat,Dinner,4
44,30.40,5.60,Male,No,Sun,Dinner,4
47,32.40,6.00,Male,No,Sun,Dinner,4
52,34.81,5.20,Female,No,Sun,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
116,29.93,5.07,Male,No,Sun,Dinner,4
155,29.85,5.14,Female,No,Sun,Dinner,5
170,50.81,10.00,Male,Yes,Sat,Dinner,3
172,7.25,5.15,Male,Yes,Sun,Dinner,2
181,23.33,5.65,Male,Yes,Sun,Dinner,2


## 🔴 Part 3: 聚合与分组 (GROUP BY)

**对应 SQL**：`GROUP BY`

### 练习 3.1：谁给的小费多？
按 `sex` (性别) 分组，计算 `tip` (小费) 的平均值。

In [28]:
# ✏️ 语法提示: df.groupby('分组列')['计算列'].mean()

tips.groupby('sex')['tip'].mean()


/var/folders/35/q6rh83x91bzgf3gcsb_13f_80000gn/T/ipykernel_98260/1441351526.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby('sex')['tip'].mean()


sex
Male      3.089618
Female    2.833448
Name: tip, dtype: float64

In [ ]:
# 🔑 参考答案
tips.groupby('sex')['tip'].mean()

### 练习 3.2：多维度聚合 (Complex Agg)
我们想看不同 `day` 的销售情况：
1.  总共有多少单？(`count`)
2.  总销售额是多少？(`sum`)
3.  最高一单是多少？(`max`)

In [30]:
# ✏️ 语法提示: df.groupby('day')['total_bill'].agg(['count', 'sum', 'max'])
tips.groupby('day')['total_bill'].agg(['count','nunique', 'sum', 'max'])


/var/folders/35/q6rh83x91bzgf3gcsb_13f_80000gn/T/ipykernel_98260/3277141805.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby('day')['total_bill'].agg(['count','nunique', 'sum', 'max'])


,count,nunique,sum,max
day,,,,
Thur,62,61,1096.33,43.11
Fri,19,18,325.88,40.17
Sat,87,85,1778.40,50.81
Sun,76,76,1627.16,48.17


## 🔵 Part 4: 终极实战 - 新增列 (Creating Columns)

**对应 SQL**：`SELECT a / b AS ratio`

**任务**：
我们想知道谁最“大方”。
请计算 **小费占比** (`tip_rate` = `tip` / `total_bill`)，并把它作为新的一列加到 `tips` 表里。

In [35]:
# ✏️ 在这里写你的代码
# tips['tip_rate'] = ...
tips['tip_rate'] = tips['tip'] / tips['total_bill']
# 检查一下前5行，看看新列有没有加上
tips.head()

,total_bill,tip,sex,smoker,day,time,size,tip_rate
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808
